# boolean-mask-combine — ex1: five-predicate ray-triangle inside test

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `boolean-mask-combine`. Running the final beacon cell reports progress against the `Numpy: Boolean mask combine` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Boolean mask combine` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`boolean-mask-combine`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "boolean-mask-combine"
DD_SUBTOPIC = "Numpy: Boolean mask combine"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Combining boolean masks — quick refresher

Multi-criterion selection is the bread-and-butter of vectorised filtering. PyTorch / NumPy give you three logical operators on `bool` tensors:

- `&` — elementwise AND
- `|` — elementwise OR
- `~` — elementwise NOT (logical, on `bool` tensors)

All three are **elementwise** and broadcast normally. They produce a fresh `bool` tensor; the operands are not consumed.

**Parenthesise every comparison.** `&` / `|` bind tighter than `<` / `>` / `==` in Python, so `x > 0 & x < 10` parses as `x > (0 & x) < 10` — a silent disaster. Always write `(x > 0) & (x < 10)`.

**Watch the dtype.** The operands must be `bool`. If you have `0/1` integer flags, convert with `.bool()` first or you'll get bitwise math on integers (`& == bitwise-and`, not logical-and).

**Canonical ARENA pattern.** The ray-triangle inside test ANDs five predicates: `(s >= 0) & (u >= 0) & (v >= 0) & (u + v <= 1) & ~is_singular`. Five comparisons → five bool tensors → ANDed down to one final mask.

### Exercise 1 — five-predicate ray-triangle inside test

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply elementwise `&`, `|`, `~` on bool tensors to combine five comparison-derived predicates into the single 'point inside triangle and matrix not singular' mask used by ARENA's batched ray-triangle intersection.
> Keywords: and, or, not, ray-tracing, inside-test
> ```

**KCs targeted:** `mask-bitwise-and-or`, `mask-parenthesize-comparisons`

Implement `ex1_inside_test(s, u, v, is_singular)`.

Given the per-(ray, triangle) outputs of `t.linalg.solve` on ARENA's Moller-Trumbore matrix:
- `s`: `(NR, NT)` ray parameter (signed distance along the ray).
- `u, v`: `(NR, NT)` barycentric coords.
- `is_singular`: `(NR, NT)` bool — True where the 3x3 was singular and the solve output is garbage.

Return a `(NR, NT)` bool mask that is `True` iff:
1. `s >= 0` (intersection in front of the ray origin)
2. `u >= 0`
3. `v >= 0`
4. `u + v <= 1` (inside the triangle's barycentric region)
5. NOT `is_singular`

**Critical syntax point.** Each comparison MUST be parenthesised — `&` binds tighter than `>=` in Python, so `s >= 0 & u >= 0` silently parses as `s >= (0 & u) >= 0`. Write `(s >= 0) & (u >= 0) & ...`.

In [ ]:
def ex1_inside_test(s: Tensor, u: Tensor, v: Tensor, is_singular: Tensor) -> Tensor:
    """Combine 5 predicates → (NR, NT) bool intersect mask."""
    raise NotImplementedError()


def _test_ex1():
    # Hand-build 6 (ray, tri) slots — one for each failure mode + one all-True.
    s   = t.tensor([[ 0.5, -0.1,  0.5,  0.5,  0.5,  0.5]])
    u   = t.tensor([[ 0.3,  0.3, -0.1,  0.3,  0.6,  0.3]])
    v   = t.tensor([[ 0.3,  0.3,  0.3,  -0.1, 0.6,  0.3]])
    sng = t.tensor([[False, False, False, False, False, True]])
    out = ex1_inside_test(s, u, v, sng)
    # Slot 0: all pass → True. Slots 1-5: each fails one predicate.
    expected = t.tensor([[True, False, False, False, False, False]])
    assert out.dtype == t.bool, f'expected bool, got {out.dtype}'
    assert out.shape == (1, 6), f'expected (1,6), got {tuple(out.shape)}'
    assert t.equal(out, expected), f'mismatch:\n  got      {out}\n  expected {expected}'

    # --- Random batch — must agree with the long-form (((... & ...) & ...) & ...) ---
    rng = t.Generator().manual_seed(11)
    NR, NT = 7, 9
    s2 = t.randn(NR, NT, generator=rng)
    u2 = t.randn(NR, NT, generator=rng) * 0.6
    v2 = t.randn(NR, NT, generator=rng) * 0.6
    sng2 = t.randn(NR, NT, generator=rng) > 1.5  # ~6% True
    ground = (s2 >= 0) & (u2 >= 0) & (v2 >= 0) & (u2 + v2 <= 1) & (~sng2)
    assert t.equal(ex1_inside_test(s2, u2, v2, sng2), ground), 'random-batch mask mismatch'

    # --- Edge: all singular → all False regardless of other predicates ---
    all_sng = t.ones(NR, NT, dtype=t.bool)
    assert not ex1_inside_test(s2, u2, v2, all_sng).any().item(), 'all-singular must yield all-False'

    # --- Edge: trivially passing case (s=u=v=0, not singular) → boundary hit, expect True ---
    zero = t.zeros(2, 2)
    ok = t.zeros(2, 2, dtype=t.bool)
    assert ex1_inside_test(zero, zero, zero, ok).all().item(), '(s=u=v=0, ~singular) lies on the boundary → must be True'
    print(f'random-batch: {ground.sum().item()}/{NR*NT} slots passed all 5 predicates')
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_inside_test(s: Tensor, u: Tensor, v: Tensor, is_singular: Tensor) -> Tensor:
    return (s >= 0) & (u >= 0) & (v >= 0) & (u + v <= 1) & (~is_singular)
```

**Order of operations.** `&` chains left-to-right; `a & b & c & d & e` is `((((a & b) & c) & d) & e)`. Python doesn't short-circuit elementwise — every predicate is fully evaluated, then the bitwise AND ladder collapses them.

**Why parenthesise every comparison.** `&` and `|` are bitwise operators in Python, with precedence *higher* than `<`, `>`, `==`. So `s >= 0 & u >= 0` parses as `s >= (0 & u) >= 0` — which is at best wrong and at worst silently runs on int tensors. The parens around each comparison are mandatory.

**Why `~` not `not`.** `not` calls `__bool__()` which only works on a 0-D tensor (and raises on a >0-D one). `~` is the elementwise bitwise NOT, which on `bool` tensors is the logical NOT.

**Performance note.** Each `&` allocates a new bool tensor. If memory is tight on long predicate chains, build incrementally: `mask = s >= 0; mask &= u >= 0; ...` (`&=` is in-place).
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()